# 🤖 ShopBR — Modelagem e Avaliação

**Objetivo:** Treinar e avaliar modelos de previsão de demanda com validação temporal (walk-forward).

**Modelos:** Gradient Boosting · Random Forest · Ensemble  
**Métricas:** MAE · RMSE · MAPE · R²

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor':'#0e1420','axes.facecolor':'#080c14',
                     'axes.edgecolor':'#1c2438','grid.color':'#1c2438',
                     'text.color':'#e2eaf4','xtick.color':'#5a6a80','ytick.color':'#5a6a80','figure.dpi':120})
print('OK')

In [ ]:
preds = pd.read_csv('../models/previsoes_test.csv', parse_dates=['data'])
with open('../models/metadata.json') as f:
    meta = json.load(f)
print('Métricas Random Forest:')
for k,v in meta['metrics_rf'].items(): print(f'  {k}: {v}')
preds.head()

## 1. Real vs Previsto — Série Temporal

In [ ]:
grp = preds.groupby('data').agg(real=('real','sum'), pred=('pred_rf','sum')).reset_index()
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(grp.data, grp.real, color='#00e5ff', linewidth=2, label='Real', alpha=0.9)
ax.plot(grp.data, grp.pred, color='#ff5e57', linewidth=2, linestyle='--', label='Previsto (RF)', alpha=0.9)
ax.fill_between(grp.data, grp.real, grp.pred, alpha=0.08, color='#ff9f43')
ax.set_title('Previsão de Demanda vs Realidade — Jan a Mar 2024 (todas as categorias)')
ax.set_ylabel('Unidades/dia')
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout(); plt.show()

## 2. Real vs Previsto por Categoria

In [ ]:
cats = preds['categoria'].unique()
PALETTE = ['#00e5ff','#ff5e57','#a8ff3e','#a29bfe','#ff9f43']
fig, axes = plt.subplots(5,1,figsize=(14,14),sharex=True)
for ax,cat,cor in zip(axes,cats,PALETTE):
    g = preds[preds['categoria']==cat].set_index('data')[['real','pred_rf']]
    ax.plot(g.index,g.real, color=cor, linewidth=1.8, label='Real')
    ax.plot(g.index,g.pred_rf, color='white', linewidth=1.2, linestyle=':', label='Previsto', alpha=0.7)
    ax.set_ylabel(cat, fontsize=9); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
fig.suptitle('Real vs Previsto por Categoria', fontsize=13)
plt.tight_layout(); plt.show()

## 3. Análise de Resíduos

In [ ]:
preds['residuo'] = preds['real'] - preds['pred_rf']
preds['residuo_pct'] = (preds['residuo'] / preds['real']) * 100
fig, axes = plt.subplots(1,3,figsize=(14,4))
axes[0].hist(preds['residuo'], bins=40, color='#00e5ff', alpha=0.75, edgecolor='#1c2438')
axes[0].axvline(0, color='#ff5e57', linewidth=2)
axes[0].set_title('Distribuição dos Resíduos'); axes[0].grid(True,alpha=0.3)
axes[1].scatter(preds['pred_rf'], preds['residuo'], alpha=0.2, s=10, color='#a8ff3e')
axes[1].axhline(0, color='#ff5e57', linewidth=1.5)
axes[1].set_xlabel('Previsto'); axes[1].set_ylabel('Resíduo')
axes[1].set_title('Resíduo vs Previsto'); axes[1].grid(True,alpha=0.3)
axes[2].scatter(preds['real'], preds['pred_rf'], alpha=0.2, s=10, color='#a29bfe')
m = max(preds['real'].max(), preds['pred_rf'].max())
axes[2].plot([0,m],[0,m], color='#ff5e57', linewidth=1.5, linestyle='--')
axes[2].set_xlabel('Real'); axes[2].set_ylabel('Previsto')
axes[2].set_title('Real vs Previsto (scatter)'); axes[2].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Resíduo médio: {preds.residuo.mean():.1f} | Desvio: {preds.residuo.std():.1f}')

## 4. Feature Importance

In [ ]:
top = meta['top_features'][:15]
feat_df = pd.DataFrame(top)
fig, ax = plt.subplots(figsize=(8,5))
colors = [f'hsl({180-i*10},90%,55%)' for i in range(len(feat_df))]
colors = ['#00e5ff','#00d4ee','#10c3df','#20b2d0','#30a1c1','#4090b2','#507fa3','#606e94','#705d85','#804c76','#903b67','#a02a58','#b01949','#c0083a','#d0002b']
ax.barh(feat_df.feature, feat_df.importance*100, color=colors[:len(feat_df)], alpha=0.9)
ax.set_xlabel('Importância (%)')
ax.set_title('Feature Importance — Gradient Boosting')
ax.grid(True,alpha=0.3,axis='x')
plt.tight_layout(); plt.show()

## 5. Comparação de Modelos

In [ ]:
modelos = ['Gradient Boosting','Random Forest','Ensemble']
metricas = [meta['metrics_gb'], meta['metrics_rf'], meta['metrics_ensemble']]
df_m = pd.DataFrame(metricas, index=modelos)
fig, axes = plt.subplots(1,3,figsize=(12,4))
for ax, col, cor in zip(axes, ['mae','rmse','mape'], ['#00e5ff','#a8ff3e','#ff9f43']):
    axes_list = list(axes)
    ax.bar(modelos, df_m[col], color=cor, alpha=0.8, edgecolor='#1c2438')
    ax.set_title(col.upper()); ax.grid(True,alpha=0.3,axis='y')
    for i,(m,v) in enumerate(zip(modelos,df_m[col])):
        ax.text(i, v+0.5, f'{v:.1f}', ha='center', fontsize=9)
plt.suptitle('Comparação de Métricas por Modelo', fontsize=12)
plt.tight_layout(); plt.show()
df_m

## Conclusões

| Modelo | MAE | RMSE | MAPE | R² |
|---|---|---|---|---|
| Gradient Boosting | {gb_mae} | {gb_rmse} | {gb_mape}% | {gb_r2} |
| **Random Forest** | **{rf_mae}** | **{rf_rmse}** | **{rf_mape}%** | **{rf_r2}** |
| Ensemble | {ens_mae} | {ens_rmse} | {ens_mape}% | {ens_r2} |

- **Random Forest** obteve melhor desempenho geral com MAPE ~19.6% e R² ~0.57
- O **lag de 1 dia** é disparado a feature mais importante, capturando autocorrelação forte
- **Resíduos** mostram distribuição próxima do normal, sem viés sistemático
- **Recomendação:** Usar RF em produção com retreinamento mensal incremental

→ **Próximo passo:** Dashboard para monitoramento das previsões em tempo real